# Lesson 07 - پلاننگ ڈیزائن پیٹرن

یہ نوٹ بک مائیکروسافٹ ایجنٹ فریم ورک کا استعمال کرتے ہوئے AI ایجنٹس کے لیے **پلاننگ ڈیزائن پیٹرن** کو ظاہر کرتی ہے۔
آپ سیکھیں گے کہ پیچیدہ سفر کی درخواستوں کو منظم ذیلی کاموں میں کیسے توڑا جائے، انہیں ماہر ایجنٹس کو تفویض کیا جائے،
اور نتیجے میں بننے والے منصوبے کو کیسے نافذ کیا جائے — یہ سب Pydantic ماڈلز کی مدد سے منظم آؤٹ پٹ کے ذریعے۔


## سیٹ اپ


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os, asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ٹاسک تقسیم

ٹاسک تقسیم منصوبہ بندی کے ڈیزائن پیٹرن کا مرکزی جزو ہے۔ ایک واحد ایجنٹ سے پیچیدہ درخواست کو ابتدا سے انتہا تک سنبھالنے کے بجائے، ہم مسئلے کو چھوٹے، اچھی طرح سے وضاحتی **ذیلی کاموں** میں تقسیم کرتے ہیں۔ ہر ذیلی کام کسی ماہر ایجنٹ کو تفویض کیا جاتا ہے (مثلاً، فلائٹس، ہوٹلز، سرگرمیاں) واضح ترجیحات اور انحصار کی ترتیب کے ساتھ۔

یہ طریقہ کئی فوائد فراہم کرتا ہے:
- **وضاحت**: ہر ذیلی کام کی ایک ہی ذمہ داری ہوتی ہے۔
- **ہم زمانی**: آزاد ذیلی کام بیک وقت چل سکتے ہیں۔
- **اعتبار**: ناکامیاں انفرادی ذیلی کاموں تک محدود رہتی ہیں۔
- **بجٹ کا تعاقب**: اخراجات ہر ذیلی کام کے لئے تخمینہ لگائے جاتے ہیں اور مجموعی کیے جاتے ہیں۔


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## منظم شدہ آؤٹ پٹ کے ساتھ ایک پلاننگ ایجنٹ بنانا

پلاننگ ایجنٹ بطور **فرنٹ ڈیسک کوآرڈینیٹر** کام کرتا ہے۔ ایک اعلی سطح کی سفر کی درخواست دی گئی ہے تو یہ ایک منظم `TravelPlan` تیار کرتا ہے — درخواست کو چھوٹے کاموں میں تقسیم کرنا، ترجیحات مقرر کرنا، اور انحصارات کی نشاندہی کرنا تاکہ ایک کنسیئر یا عمل درآمد کی سطح کام انجام دے سکے۔


In [ ]:
planning_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    )
if result:
    plan = result
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## ماہر ٹولز کے ساتھ منصوبے پر عملدرآمد

جب فرنٹ ڈیسک ایجنٹ ایک منظم منصوبہ تیار کر لیتا ہے، تو **کنسیئرج ایجنٹ** اسے عملی جامہ پہنانے کا کام انجام دیتا ہے۔
ہر ماہر آلہ ذیلی کام کی ایک قسم (پروازیں، ہوٹل، سرگرمیاں) سنبھالتا ہے۔ کنسیئرج منصوبے کے ذیلی کاموں کو ان کی انحصاری کے ترتیب میں دہراتا ہے اور ہر ایک کو مناسب آلے کو بھیجتا ہے۔


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = await provider.create_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## خلاصہ

اس سبق میں آپ نے AI ایجنٹس کے لیے **پلاننگ ڈیزائن پیٹرن** سیکھا:

1. **ٹاسک کو توڑنا** — ایک فرنٹ ڈیسک پلاننگ ایجنٹ پیچیدہ سفر کی درخواست کو ساختہ ذیلی کاموں میں تقسیم کرتا ہے، جو پائیڈینٹک ماڈلز استعمال کرتے ہوئے ہر ایک کو ایک ماہر ایجنٹ کو ترجیحات اور انحصار کے ساتھ تفویض کرتا ہے۔
2. **ساختہ آؤٹ پٹ** — `response_format` فراہم کر کے ایجنٹ ایک آزاد متن کی بجائے ایک تصدیق شدہ `TravelPlan` آبجیکٹ واپس کرتا ہے، جس سے نیچے کی جانب پراسیسنگ قابل اعتماد ہو جاتی ہے۔
3. **منصوبہ بندی کا نفاذ** — ایک کنسیئر ایجنٹ ذیلی کاموں پر متخصص آلات (`book_flight`, `reserve_hotel`, `book_activity`) استعمال کرتے ہوئے منصوبہ عمل میں لاتا ہے اور نتائج رپورٹ کرتا ہے۔

یہ پیٹرن *کیا کرنا ہے* (پلاننگ) کو *کیسے کرنا ہے* (نفاذ) سے جدا کرتا ہے، جس سے ایجنٹس زیادہ ماڈیولر، جانچنے کے قابل، اور آسانی سے بڑھائے جا سکنے والے بن جاتے ہیں۔


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**دستبرداری**:  
یہ دستاویز AI ترجمہ سروس [Co-op Translator](https://github.com/Azure/co-op-translator) کے ذریعے ترجمہ کی گئی ہے۔ اگرچہ ہم درستگی کے لیے کوشاں ہیں، براہ کرم اس بات کا نوٹس لیں کہ خودکار ترجمے میں غلطیاں یا کمی بیشی ہو سکتی ہے۔ اصل دستاویز اپنی مادری زبان میں ہی مستند ماخذ سمجھی جانی چاہیے۔ اہم معلومات کے لیے پیشہ ور انسانی ترجمہ تجویز کیا جاتا ہے۔ اس ترجمے کے استعمال سے پیدا ہونے والے کسی بھی غلط فہم یا غلط تعبیرات کے لیے ہم ذمہ دار نہیں ہیں۔
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
